# axikernels ObsPy Workflow for a Real AxiSEM3D Run

This notebook assumes you already have an AxiSEM3D simulation on disk and want to inspect its outputs with `axikernels` and ObsPy.

To create the small demo run that this notebook targets by default, run the following command from the repository root:

```bash
./examples/run_handlers_demo.sh
```

That command creates a lightweight demo simulation with both station output in `output/stations/GSN_Station_Grid/` and element output in `output/elements/mantle/`. Then open this notebook and work through the cells below.

### What this notebook covers
1. Point the handlers at a real simulation directory
2. Load station output into `StationOutput`
3. Build ObsPy `Stream`, `Inventory`, and `Catalog` objects
4. Export and reload `obspyfied` files
5. Load element output from the same run and interpolate it to stations

### Prerequisites
```bash
conda activate axikernels_env
pip install -e .
```

## Section 1 - Locate the Repository and the Demo Run

Resolve repository-local paths and define the default location of the demo simulation created by `./examples/run_handlers_demo.sh`.

In [1]:
import sys
from pathlib import Path

if "__file__" in dir():
    search_root = Path(__file__).resolve().parent
else:
    search_root = Path.cwd().resolve()

REPO_ROOT = None
for candidate in (search_root, *search_root.parents):
    if (candidate / "axikernels" / "core").is_dir():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate the AxiSEM3D_Kernels repository root. Open the notebook inside the repository or adjust the working directory."
    )

repo_root_str = str(REPO_ROOT)
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)

RUN_SCRIPT = REPO_ROOT / "examples" / "run_handlers_demo.sh"
DEFAULT_SIM_DIR = REPO_ROOT / "examples" / "demo_runs" / "HANDLERS_EXAMPLE_RUN"

print("Repository root :", REPO_ROOT)
print("Demo runner     :", RUN_SCRIPT)
print("Default run dir :", DEFAULT_SIM_DIR)

Repository root : /home/adrian/PhD/AxiSEM3D/AxiSEM3D_Kernels
Demo runner     : /home/adrian/PhD/AxiSEM3D/AxiSEM3D_Kernels/examples/run_handlers_demo.sh
Default run dir : /home/adrian/PhD/AxiSEM3D/AxiSEM3D_Kernels/examples/demo_runs/HANDLERS_EXAMPLE_RUN


## Section 2 - Import ObsPy and the axikernels Handlers

In [2]:
import matplotlib.pyplot as plt
import obspy
import pandas as pd

from axikernels.core.handlers.obspy_output import ObspyfiedOutput
from axikernels.core.handlers.station_output import StationOutput

try:
    from axikernels.core.handlers.element_output import ElementOutput
    element_import_error = None
except Exception as exc:
    ElementOutput = None
    element_import_error = exc

print("ObsPy version   :", obspy.__version__)
print("StationOutput   : ready")
print("ObspyfiedOutput : ready")
if ElementOutput is None:
    print(f"ElementOutput   : unavailable ({type(element_import_error).__name__})")
else:
    print("ElementOutput   : ready")

ObsPy version   : 1.5.0
StationOutput   : ready
ObspyfiedOutput : ready
ElementOutput   : ready


## Section 3 - Choose the Simulation Directory

By default, the notebook uses the run directory created by `./examples/run_handlers_demo.sh`. If you want to use your own AxiSEM3D run, change `SIM_DIR`, `STATION_GROUP`, `ELEMENT_GROUP`, and `STATION_FILE_NAME` in the next cell.

In [3]:
SIM_DIR = DEFAULT_SIM_DIR
STATION_GROUP = "GSN_Station_Grid"
ELEMENT_GROUP = "mantle"
STATION_FILE_NAME = "GSN_small.txt"

STATION_FILE = SIM_DIR / "input" / STATION_FILE_NAME
STA_OUT = SIM_DIR / "output" / "stations" / STATION_GROUP
ELEM_OUT = SIM_DIR / "output" / "elements" / ELEMENT_GROUP

missing = [path for path in [SIM_DIR, STATION_FILE, STA_OUT, ELEM_OUT] if not path.exists()]
if missing:
    missing_text = "\n".join(f"  - {path}" for path in missing)
    if SIM_DIR == DEFAULT_SIM_DIR:
        raise FileNotFoundError(
            "The demo run is not ready yet. From the repository root, run:\n\n"
            "  ./examples/run_handlers_demo.sh\n\n"
            f"Missing paths:\n{missing_text}"
        )
    raise FileNotFoundError(
        "One or more paths for your selected simulation are missing. Update SIM_DIR, STATION_GROUP, ELEMENT_GROUP, or STATION_FILE_NAME and try again.\n\n"
        f"Missing paths:\n{missing_text}"
    )

print("Simulation directory :", SIM_DIR)
print("Station file         :", STATION_FILE)
print("Station output       :", STA_OUT)
print("Element output       :", ELEM_OUT)

Simulation directory : /home/adrian/PhD/AxiSEM3D/AxiSEM3D_Kernels/examples/demo_runs/HANDLERS_EXAMPLE_RUN
Station file         : /home/adrian/PhD/AxiSEM3D/AxiSEM3D_Kernels/examples/demo_runs/HANDLERS_EXAMPLE_RUN/input/GSN_small.txt
Station output       : /home/adrian/PhD/AxiSEM3D/AxiSEM3D_Kernels/examples/demo_runs/HANDLERS_EXAMPLE_RUN/output/stations/GSN_Station_Grid
Element output       : /home/adrian/PhD/AxiSEM3D/AxiSEM3D_Kernels/examples/demo_runs/HANDLERS_EXAMPLE_RUN/output/elements/mantle


## Section 4 - Load Station Output

Start with the station output folder from the finished AxiSEM3D run. This is the most common entry point for a user who wants to work with ObsPy objects.

In [4]:
so = StationOutput(str(STA_OUT))

print("StationOutput loaded")
print("  Simulation name :", so.simulation_name)
print("  Station group   :", so.station_group_name)
print("  Simulation root :", so.path_to_simulation)

StationOutput loaded
  Simulation name : HANDLERS_EXAMPLE_RUN
  Station group   : GSN_Station_Grid
  Simulation root : /home/adrian/PhD/AxiSEM3D/AxiSEM3D_Kernels/examples/demo_runs/HANDLERS_EXAMPLE_RUN


## Section 5 - Inspect Stations, Channels, and Time Axis

The station file tells you which stations were requested in the simulation. The handler adds the available wavefield channels and time axis from the generated output files.

In [5]:
stations = pd.read_csv(
    STATION_FILE,
    sep=r"\s+",
    header=None,
    names=["name", "network", "latitude", "longitude", "useless", "depth"],
    comment="#",
)

print("First stations in the station file:")
print(stations[["network", "name", "latitude", "longitude", "depth"]].head(8).to_string(index=False))
print()
print("Coordinate frame :", so.coordinate_frame)
print("Channels (yaml)  :", so.channels)
print("Detailed channels:", so.detailed_channels)
print()
print("Time axis")
print(f"  n_samples : {len(so.data_time)}")
print(f"  t_start   : {so.data_time[0]:.2f} s")
print(f"  t_end     : {so.data_time[-1]:.2f} s")
print(f"  dt        : {so.data_time[1] - so.data_time[0]:.4f} s")
print()
print("Rank list preview:")
print(so._rank_list.head(8).to_string(index=False))

First stations in the station file:
network name  latitude  longitude  depth
     II  AAK   42.6390    74.4940   30.0
     II ABKT   37.9304    58.1189    7.0
     II ABPO  -19.0180    47.2290    5.3
     II  ALE   82.5033   -62.3500    0.0
     II  ARU   56.4302    58.5625    0.0
     II ASCN   -7.9327   -14.3601  100.0
     II  BFO   48.3319     8.3311    0.0
     II BORG   64.7474   -21.3268   95.0

Coordinate frame : RTZ
Channels (yaml)  : ['U']
Detailed channels: ['UR', 'UT', 'UZ']

Time axis
  n_samples : 1115
  t_start   : -123450000000000.00 s
  t_end     : -123450000000000.00 s
  dt        : 0.0000 s

Rank list preview:
 MPI_RANK STATION_KEY  STATION_INDEX_IN_RANK
        0      II.AAK                      0
        0     II.ABKT                      1
        0     II.ABPO                      2
        0      II.ALE                      3
        0      II.ARU                      4
        0     II.ASCN                      5
        0      II.BFO                      6
   

## Section 6 - Build ObsPy Objects from Station Output

Pick one or two stations, load the waveforms, and inspect the ObsPy objects created by the handler.

In [ ]:
example_network = stations.iloc[0]["network"]
example_station = stations.iloc[0]["name"]
if len(stations) > 1:
    second_network = stations.iloc[1]["network"]
    second_station = stations.iloc[1]["name"]
else:
    second_network = example_network
    second_station = example_station

raw = so.load_data_at_station(example_network, example_station)
stream = so.stream([example_network, second_network], [example_station, second_station])
inv = so.inventory
cat = so.catalogue

print(f"Waveform array for {example_network}.{example_station}: {raw.shape}")
print()
print("ObsPy Stream:")
print(stream)
print()
print("Inventory summary:")
print(f"  Networks : {len(inv.networks)}")
print(f"  Stations : {sum(len(net.stations) for net in inv.networks)}")
print()
print("Catalog summary:")
print(f"  Events   : {len(cat)}")
if len(cat) > 0:
    origin = cat[0].preferred_origin() or cat[0].origins[0]
    print(f"  Origin   : lat={origin.latitude}, lon={origin.longitude}, depth_km={origin.depth / 1000:.1f}")

## Section 7 - Plot One Station

Plot all available components for a single station so you can quickly sanity-check the output.

In [ ]:
t = so.data_time
n_channels = raw.shape[0]

fig, axes = plt.subplots(n_channels, 1, figsize=(11, 1.8 * n_channels + 1.2), sharex=True)
if n_channels == 1:
    axes = [axes]

fig.suptitle(f"Waveforms at {example_network}.{example_station} ({so.coordinate_frame})", fontsize=12)

for ax, data_row, label in zip(axes, raw, so.detailed_channels):
    ax.plot(t, data_row, lw=0.9, color="steelblue")
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
    ax.axhline(0.0, color="k", lw=0.5, ls="--")

axes[-1].set_xlabel("Time [s]")
plt.tight_layout()
plt.show()

## Section 8 - Export Station Output to ObsPy Files

If you want portable ObsPy files for later work, `so.obspyfy()` writes MiniSEED, StationXML, and QuakeML into an `obspyfied/` folder under the station group output.

In [ ]:
so.obspyfy()

obspy_dir = STA_OUT / "obspyfied"
written = sorted(obspy_dir.iterdir())

print("Files written to obspyfied:")
for path in written:
    print(f"  {path.name}")

## Section 9 - Reload the Exported ObsPy Files

`ObspyfiedOutput` is the simplest way to reopen the MiniSEED, StationXML, and QuakeML files produced by `so.obspyfy()`.

In [ ]:
oo = ObspyfiedOutput(obspyfied_path=str(obspy_dir))

print("Reloaded ObspyfiedOutput")
print(f"  Trace count    : {len(oo.stream)}")
print(f"  Inventory size : {sum(len(net.stations) for net in oo.inv.networks)} stations")
print(f"  Event count    : {len(oo.cat)}")

## Section 10 - Load Element Output From the Same Simulation

The same AxiSEM3D run also produced an element-output group. This section reuses the existing station file from the run input directory, so the workflow stays read-only with respect to the simulation tree.

In [ ]:
if ElementOutput is None:
    print("ElementOutput could not be imported in this environment:")
    print(f"  {element_import_error}")
else:
    eo = ElementOutput(str(ELEM_OUT))
    element_stream = eo.stream_STA(str(STATION_FILE), channels=["UR", "UT", "UZ"])
    element_inv = eo.create_inventory(str(STATION_FILE))

    print("ElementOutput loaded")
    print("  Element groups :", eo.element_groups)
    print("  Source lat/lon :", eo.source_lat, eo.source_lon)
    print("  Source depth m :", eo.source_depth)
    print()
    print(f"Interpolated stream traces : {len(element_stream)}")
    print(f"Interpolated inventory size: {sum(len(net.stations) for net in element_inv.networks)} stations")
    print(f"Station file used          : {STATION_FILE}")

## Section 11 - When to Use Each Handler

| Task | Handler | Typical input path |
|------|---------|--------------------|
| Read station recordings exactly as written by AxiSEM3D | `StationOutput` | `output/stations/<group>/` |
| Reload previously exported MiniSEED + StationXML + QuakeML | `ObspyfiedOutput` | `output/stations/<group>/obspyfied/` |
| Interpolate element output to chosen stations | `ElementOutput` | `output/elements/<group>/` |

In practice, most users start with `StationOutput` and only move to `ElementOutput` when they need wavefields away from the original station list.

## Section 12 - Quick Summary

The next cell prints the key paths and objects created in this notebook so you can quickly confirm what you now have available for further analysis.

In [ ]:
print("Simulation directory:")
print(f"  {SIM_DIR}")
print()
print("Key output directories:")
print(f"  Station output : {STA_OUT}")
print(f"  Element output : {ELEM_OUT}")
print(f"  Obspyfied      : {obspy_dir}")
print()
print("Objects created in this notebook:")
print(f"  StationOutput    : {type(so).__name__}")
print(f"  ObsPy Stream     : {type(stream).__name__} with {len(stream)} traces")
print(f"  Inventory        : {type(inv).__name__}")
print(f"  Catalog          : {type(cat).__name__}")
print(f"  ObspyfiedOutput  : {type(oo).__name__}")
if 'element_stream' in globals():
    print(f"  Element stream   : {type(element_stream).__name__} with {len(element_stream)} traces")
else:
    print("  Element stream   : not created in this environment")

## Section 13 - Using Your Own AxiSEM3D Run

Once you are comfortable with the demo, you can reuse the same notebook on your own simulation.

Replace these values in Section 3:

- `SIM_DIR`
- `STATION_GROUP`
- `ELEMENT_GROUP`
- `STATION_FILE_NAME`

The rest of the workflow stays the same:

1. load `StationOutput` from `output/stations/<group>/`
2. inspect metadata and build ObsPy objects
3. export and reload `obspyfied` files if useful
4. load `ElementOutput` from `output/elements/<group>/` when you need interpolated wavefields

In [ ]:
print("Minimal post-processing pattern")
print("=" * 32)
print(
    """from axikernels.core.handlers.station_output import StationOutput
from axikernels.core.handlers.element_output import ElementOutput

sim_dir = \"<simulation_dir>\"
station_group = \"<station_group>\"
element_group = \"<element_group>\"

so = StationOutput(f\"{sim_dir}/output/stations/{station_group}\")
stream = so.stream([\"NET\"], [\"STA\"])
inv = so.inventory
cat = so.catalogue
so.obspyfy()

eo = ElementOutput(f\"{sim_dir}/output/elements/{element_group}\")
element_stream = eo.stream_STA(
    f\"{sim_dir}/input/STATIONS.txt\",
    channels=[\"UR\", \"UT\", \"UZ\"],
)"""
)